<a href="https://colab.research.google.com/github/2303a51546/2303A51546_B10-RL/blob/main/Copy_of_Lab_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def env_state():
    return ["S0", "S1"]

states = env_state()
terminal_state = "G"
all_states = states + [terminal_state]
print("=" * 60)
print("1. STATES")
print("=" * 60)
print("Non-terminal states:", states)
print("Terminal state:", terminal_state)
print("All states:", all_states)
def env_actions():
    return ["up", "down", "left", "Right"]

actions = env_actions()

print("\n" + "=" * 60)
print("2. ACTIONS")
print("=" * 60)
print("Actions:", actions)
def env_reward():
    return {
        "move": -1,
        "goal": 10,
        "obs": -1
    }

rewards = env_reward()

print("\n" + "=" * 60)
print("3. REWARDS")
print("=" * 60)
print("Normal movement reward:", rewards["move"])
print("Goal reward:", rewards["goal"])
print("Wall/obstacle reward:", rewards["obs"])
def env_df():
    gamma = 0.9
    return gamma

gamma = env_df()
theta = 1e-8
max_iterations = 100

print("\n" + "=" * 60)
print("4. DISCOUNT FACTOR AND THRESHOLD")
print("=" * 60)
print("Discount factor gamma:", gamma)
print("Convergence threshold theta:", theta)
def env_transition():
    return {
        ("S0", "up"): "S0",
        ("S0", "down"): "S1",
        ("S0", "left"): "S0",
        ("S0", "Right"): "S0",

        ("S1", "up"): "S0",
        ("S1", "down"): "S1",
        ("S1", "left"): "S1",
        ("S1", "Right"): "G"
    }

transition = env_transition()

print("\n" + "=" * 60)
print("5. TRANSITION MAPPING")
print("=" * 60)

for key, next_state in transition.items():
    print(f"{key[0]} --{key[1]}--> {next_state}")
def get_reward(state, action, next_state):
    if next_state == terminal_state:
        return rewards["goal"]
    return rewards["move"]

print("\n" + "=" * 60)
print("6. TRANSITION VERIFICATION")
print("=" * 60)

for state in states:
    for action in actions:
        assert (state, action) in transition

        next_state = transition[(state, action)]
        reward = get_reward(state, action, next_state)

        print(
            f"Current State: {state:2s} | "
            f"Action: {action:5s} | "
            f"Next State: {next_state:2s} | "
            f"Reward: {reward:+d}"
        )

print("\nAll state-action pairs have defined transitions.")
V = {state: 0.0 for state in all_states}
print("\n" + "=" * 60)
print("7. INITIAL STATE VALUES")
print("=" * 60)
for state in all_states:
    print(f"V0({state}) = {V[state]:.6f}")
print("\n" + "=" * 75)
print("8. VALUE ITERATION")
print("=" * 75)

print(
    f"{'Iteration':>10} "
    f"{'V(S0)':>14} "
    f"{'V(S1)':>14} "
    f"{'V(G)':>14} "
    f"{'Delta':>14}"
)

print(
    f"{0:10d} "
    f"{V['S0']:14.6f} "
    f"{V['S1']:14.6f} "
    f"{V['G']:14.6f} "
    f"{'-':>14}"
)

iteration_history = [V.copy()]

for iteration in range(1, max_iterations + 1):
    old_V = V.copy()
    new_V = old_V.copy()

    for state in states:
        q_values = {}
        for action in actions:
            next_state = transition[(state, action)]
            reward = get_reward(state, action, next_state)
            q_values[action] = (
                reward + gamma * old_V[next_state]
            )

        new_V[state] = max(q_values.values())

    new_V[terminal_state] = 0.0
    delta = max(
        abs(new_V[state] - old_V[state])
        for state in states
    )

    V = new_V
    iteration_history.append(V.copy())

    print(
        f"{iteration:10d} "
        f"{V['S0']:14.6f} "
        f"{V['S1']:14.6f} "
        f"{V['G']:14.6f} "
        f"{delta:14.8f}"
    )

    if delta < theta:
        print(f"\nConverged after {iteration} iterations.")
        break

else:
    raise RuntimeError("Value iteration did not converge.")


print("\n" + "=" * 60)
print("9. FINAL OPTIMAL STATE VALUES")
print("=" * 60)

for state in all_states:
    print(f"V*({state}) = {V[state]:.6f}")



def calculate_q(state, action, state_values):
    next_state = transition[(state, action)]
    reward = get_reward(state, action, next_state)

    return reward + gamma * state_values[next_state]


Q = {}

print("\n" + "=" * 60)
print("10. FINAL ACTION VALUES Q*(s, a)")
print("=" * 60)

for state in states:
    Q[state] = {}

    print(f"\nState {state}:")

    for action in actions:
        next_state = transition[(state, action)]
        reward = get_reward(state, action, next_state)

        Q[state][action] = calculate_q(
            state, action, V
        )

        print(
            f"Q*({state}, {action}) = "
            f"{reward} + {gamma} * V({next_state}) "
            f"= {Q[state][action]:.6f}"
        )


print("\n" + "=" * 60)
print("11. ACTION-VALUE TABLE")
print("=" * 60)

print(
    f"{'State':<10}"
    + "".join(f"{action:>12}" for action in actions)
)

for state in states:
    print(
        f"{state:<10}"
        + "".join(
            f"{Q[state][action]:12.4f}"
            for action in actions
        )
    )


optimal_policy = {}

print("\n" + "=" * 60)
print("12. OPTIMAL POLICY")
print("=" * 60)

for state in states:
    best_value = max(Q[state].values())

    best_actions = [
        action
        for action in actions
        if abs(Q[state][action] - best_value) < 1e-10
    ]

    optimal_policy[state] = best_actions

    print(
        f"Policy({state}) = {best_actions}, "
        f"Best Q-value = {best_value:.6f}"
    )

print("Policy(G) = Terminal state; no action required.")
print("\n" + "=" * 75)
print("13. BELLMAN OPTIMALITY VERIFICATION")
print("=" * 75)

print(
    f"{'State':<10}"
    f"{'V*(s)':>14}"
    f"{'max Q*(s,a)':>18}"
    f"{'Difference':>16}"
    f"{'Result':>12}"
)

for state in states:
    max_q = max(Q[state].values())
    difference = abs(V[state] - max_q)

    result = "PASS" if difference < 1e-6 else "FAIL"

    print(
        f"{state:<10}"
        f"{V[state]:14.6f}"
        f"{max_q:18.6f}"
        f"{difference:16.10f}"
        f"{result:>12}"
    )

    assert difference < 1e-6

print("\nBellman Optimality verified successfully.")
print("\n" + "=" * 60)
print("14. FINAL RESULT")
print("=" * 60)

print("Optimal state values:")
for state in states:
    print(f"V*({state}) = {V[state]:.6f}")

print("\nOptimal policy:")
for state in states:
    print(f"{state} -> {optimal_policy[state]}")

print("\nGoal state G is terminal, with V(G) = 0.")
print("Value Iteration completed successfully.")


1. STATES
Non-terminal states: ['S0', 'S1']
Terminal state: G
All states: ['S0', 'S1', 'G']

2. ACTIONS
Actions: ['up', 'down', 'left', 'Right']

3. REWARDS
Normal movement reward: -1
Goal reward: 10
Wall/obstacle reward: -1

4. DISCOUNT FACTOR AND THRESHOLD
Discount factor gamma: 0.9
Convergence threshold theta: 1e-08

5. TRANSITION MAPPING
S0 --up--> S0
S0 --down--> S1
S0 --left--> S0
S0 --Right--> S0
S1 --up--> S0
S1 --down--> S1
S1 --left--> S1
S1 --Right--> G

6. TRANSITION VERIFICATION
Current State: S0 | Action: up    | Next State: S0 | Reward: -1
Current State: S0 | Action: down  | Next State: S1 | Reward: -1
Current State: S0 | Action: left  | Next State: S0 | Reward: -1
Current State: S0 | Action: Right | Next State: S0 | Reward: -1
Current State: S1 | Action: up    | Next State: S0 | Reward: -1
Current State: S1 | Action: down  | Next State: S1 | Reward: -1
Current State: S1 | Action: left  | Next State: S1 | Reward: -1
Current State: S1 | Action: Right | Next State: G  | Re